# Sub-clustering in Scanpy

In [ ]:
# This cell is labelled 'parameters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3] # This takes a while per res
var_genes = 6000
n_pcs = 50
batch_col = 'sample' 

### Load Data

In [ ]:

# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
subcluster_dir = scanpy_dir + f'subclustering/' # maybe add to init_env.py later
os.makedirs(subcluster_dir, exist_ok=True)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters.h5ad')

### Set Level 1 - Cluster IDs

In [ ]:
logger.info("Setting cluster IDs ...")
# Set cluster names
cluster_anns = {
    
    '0': 'ExN-UL',
    '1': 'ExN-DL',
    '2': 'RG',
    '3': 'InN',
    '4': 'Endo-Peri',
    '5': 'OPC',
    '6': 'MG'
}

custom_palette = {
    'RG': '#FF5959',
    'ExN-UL': '#00B6EB',
    'InN': '#3CBB75FF',
    'ExN-DL': '#CEE5FD',
    'Endo-Peri': '#B200ED',
    'MG': '#F58231',
    'OPC': '#FDE725FF'
}

# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

# Check the new column
print(adata.obs[['leiden_harmony_0.1', 'cell_type']].head())

### Restore raw counts for subclustering

In [ ]:
#adata.X = get_raw_counts_all_genes(adata)  # This will set adata.X to the raw counts for all genes
adata = adata.raw.to_adata() # Bring all genes back
logger.info(f"Check raw shape ... \n\n{adata.shape}")
adata.X[:10, :10].todense()

### Subset Scanpy object by cluster

In [ ]:
# Subset for each cell type
ul_exn_adata = adata[adata.obs['cell_type'] == 'ExN-UL'].copy()
dl_exn_adata = adata[adata.obs['cell_type'] == 'ExN-DL'].copy()
rg_adata = adata[adata.obs['cell_type'] == 'RG'].copy()
inn_adata = adata[adata.obs['cell_type'] == 'InN'].copy()

# Remove main adata to free up memory
del adata

# Check the number of cells in each subset
print(ul_exn_adata.obs['cell_type'].value_counts())
print(dl_exn_adata.obs['cell_type'].value_counts())
print(rg_adata.obs['cell_type'].value_counts())
print(inn_adata.obs['cell_type'].value_counts())

### Run Scanpy pipeline on each subset

In [ ]:
for subset_adata, cell_type in zip([ul_exn_adata, dl_exn_adata, rg_adata, inn_adata],
                                  ['ExN-UL', 'ExN-DL', 'RG', 'InN']):
    logger.info(f"Processing {cell_type} for subclustering ...")

    # Ensure raw counts are available (if not, set adata.raw = adata before subsetting in earlier code)
    #if 'raw' not in subset_adata.layers:
    #   subset_adata.raw = subset_adata  # Store raw for later if needed
    
    # Normalize and log transform
    sc.pp.normalize_total(subset_adata, target_sum=1e4)
    sc.pp.log1p(subset_adata)
    
    # Identify highly variable genes
    sc.pp.highly_variable_genes(subset_adata, n_top_genes = var_genes, flavor = "seurat_v3", batch_key = batch_col)
    
    # Scale the data
    sc.pp.scale(subset_adata, max_value=10)
    
    # Perform PCA
    sc.tl.pca(subset_adata, n_comps = n_pcs, use_highly_variable = True)
    
    # Compute the neighborhood graph
    sc.pp.neighbors(subset_adata, n_neighbors=10, n_pcs=n_pcs)
    
    # UMAP embedding
    sc.tl.umap(subset_adata)
    
    # Clustering with Leiden algorithm at multiple resolutions
    for res in resolutions:
        cluster_key = f'leiden_{cell_type}_L2__{res}'
        sc.tl.leiden(subset_adata, resolution=res, key_added=cluster_key)

    # Batch correction with Harmony
    sce.pp.harmony_integrate(subset_adata, batch_col)
    subset_adata.obsm['X_pca'] = subset_adata.obsm['X_pca_harmony']
    sc.pp.neighbors(subset_adata, n_neighbors=10, n_pcs=n_pcs)
    sc.tl.umap(subset_adata) # Overwites pre-harmony X_umap

    for res in resolutions:
        cluster_key = f'leiden_{cell_type}_L2_harmony_{res}'
        sc.tl.leiden(subset_adata, resolution=res, key_added=cluster_key)


### UMAPs

In [ ]:
# Define distinct gene lists for each cell type
genes_per_type = {
    'ExN-UL': exN_genes + ["PLXNA4", "SATB2"],
    'ExN-DL': exN_genes + ["BCL11B", "TLE4"],
    'RG': r_glia_genes + ["GLI2"], 
    'InN': inN_genes + ["ERBB4"]
}
# List of subset AnnData objects and their corresponding cell types
subsets = [
    (ul_exn_adata, 'ExN-UL'),
    (dl_exn_adata, 'ExN-DL'),
    (rg_adata, 'RG'),
    (inn_adata, 'InN')
]

for res in resolutions:
    # Create a 2x4 plot for UMAPs (row 0) and violins (row 1)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    
    for idx, (subset_adata, cell_type) in enumerate(subsets):
        # Add subcluster column with L1 prefix for this resolution
        cluster_key = f'leiden_{cell_type}_L2_harmony_{res}'
        subset_adata.obs[f'subclusters_{res}'] = subset_adata.obs[cluster_key].apply(lambda x: f'{cell_type}-{x}')
        
        # Plot UMAP for this cell type (row 0)
        ax_umap = axes[0, idx]
        sc.pl.umap(
            subset_adata,
            color=f'subclusters_{res}',
            ax=ax_umap,
            title=f'{cell_type} Subclusters: Harmony L2 {res}',
            show=False
        )
        ax_umap.set_xlabel('UMAP1')
        ax_umap.set_ylabel('UMAP2')
        
        # Plot violin for distinct genes per cell type grouped by subclusters (row 1)
        genes = genes_per_type[cell_type]
        ax_violin = axes[1, idx]
        sc.pl.violin(
            subset_adata,
            keys=genes,
            groupby=f'subclusters_{res}',
            ax=ax_violin,
            show=False
        )
        ax_violin.set_title(f'{cell_type} Expression: {res}')
    
    plt.tight_layout()
    plt.show()

### Subcluster counts

In [ ]:
# Ensure all subclusters columns are added for all resolutions
for res in resolutions:
    for subset_adata, cell_type in subsets:
        cluster_key = f'leiden_{cell_type}_L2_harmony_{res}'
        subset_adata.obs[f'subclusters_{res}'] = subset_adata.obs[cluster_key].apply(lambda x: f'{cell_type}-{x}')

# Collect value counts into a list for DataFrame
data = []
for res in resolutions:
    for subset_adata, cell_type in subsets:
        col = f'subclusters_{res}'
        counts = subset_adata.obs[col].value_counts()
        for subcluster, count in counts.items():
            data.append({
                'Resolution': res,
                'Cell_Type': cell_type,
                'Subcluster': subcluster,
                'Count': count
            })

# Create DataFrame and display
df = pd.DataFrame(data)
df

### Save

In [ ]:
#for subset_adata, cell_type in zip([ul_exn_adata, dl_exn_adata, rg_adata, inn_adata],
#                                  ['ExN-UL', 'ExN-DL', 'RG', 'InN']):
#    logger.info(f"Saving {cell_type} for h5ad file ...")
#    subset_adata.write(subcluster_dir + f'adata_{cell_type}_subclusters.h5ad')
#logger.info("All done.")